# x402 `exact` scheme with Permit2 — EURe facilitator test

Tests the **Permit2** variant of the `exact` scheme end to end against a running facilitator,
paying in **EURe** (Monerium) on Base. It is the EURe counterpart of `x402_exact.ipynb`, which
covers USDC and EURC.

**Why Permit2.** USDC and EURC implement EIP-3009 (`transferWithAuthorization`), so the buyer
signs one authorization and the facilitator moves the tokens. EURe has no EIP-3009. Instead, the
buyer signs two things, both off-chain:

1. a **Permit2 witness transfer** (`permit2Authorization`): spender is the x402 exact Permit2
   proxy (`0x4020…0001`), and the witness pins the recipient (`witness.to` = the seller);
2. an **EIP-2612 permit** letting Permit2 spend the buyer's EURe (the `eip2612GasSponsoring`
   extension). The client adds it only if the buyer's Permit2 allowance is short.

The facilitator submits both in one transaction (`settleWithPermit`), so the **buyer needs no
ETH**. The seller needs a little ETH for one EURe `approve()`, and the facilitator wallet needs ETH
for settlement and the fee pull.

**18 decimals.** EURe has 18 decimals, not 6. The 0.01 fee is `10000000000000000` atomic units,
and the seller's recommended fee approval is `10¹⁸` (1 EURe = 100 fees). The notebook reads both
from `/supported` → `facilitatorFees.assets`.

**Test EURe on Base Sepolia** comes from the Monerium sandbox
(token `0x29F37F6adCa168B79B8d9567eab9BE3fBF21db85`):
1. Sign up at <https://sandbox.monerium.dev> (sandbox KYC takes test data).
2. Link the **buyer** address on **Base Sepolia**. Linking signs a message, so import that key
   into a wallet such as MetaMask first. The buyer key is `EURE_BUYER_PRIVATE_KEY` if set in
   `x402_facilitator/.env`, else `TEST_WALLET_PRIVATE_KEY`.
3. Receive → *Simulate bank transfer* → amount → *Add money*. EURe is minted to the linked address.

Without EURe the notebook still runs to `/verify`, which then reports
`permit2_insufficient_balance`. That outcome proves the payload, signatures and contracts are all
accepted, and only the balance is missing.

**Facilitator.** Point the notebook's `FACILITATOR_URL` at the deployed facilitator
(`https://facilitator.fretchen.eu`), or run a local one from `x402_facilitator/`:
`npm run build && npm run dev` (port 8080) — the default — when testing facilitator changes.

**Dry run.** Set `DRY_RUN=1` in the environment to skip the two on-chain writes (the seller's
`approve()` and `/settle`). Everything else, including `/verify`, still runs.

In [11]:
// Setup: imports and accounts
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount } from "npm:viem@2/accounts";
import { createPublicClient, createWalletClient, http, formatUnits, formatEther, parseUnits } from "npm:viem@2";
import { base, baseSepolia } from "npm:viem@2/chains";

// The package's single .env, one level up (x402_facilitator/.env).
const env = await load({ envPath: "../.env", examplePath: null, export: true });

const toKey = (k: string) => `0x${k.replace(/^0x/, "")}` as `0x${string}`;
// The buyer is the address you linked in the Monerium sandbox. EURE_BUYER_PRIVATE_KEY selects
// it when that is not the usual test wallet; otherwise TEST_WALLET_PRIVATE_KEY pays.
const BUYER_KEY_VAR = env.EURE_BUYER_PRIVATE_KEY ? "EURE_BUYER_PRIVATE_KEY" : "TEST_WALLET_PRIVATE_KEY";
const buyerAccount = privateKeyToAccount(toKey(env[BUYER_KEY_VAR]));
const sellerAccount = privateKeyToAccount(toKey(env.NFT_WALLET_PRIVATE_KEY));
const SELLER_ADDRESS = sellerAccount.address;

// DRY_RUN=1 skips the on-chain writes (seller approve, /settle).
const DRY_RUN = Deno.env.get("DRY_RUN") === "1";

console.log("🚀 x402 exact (Permit2) — EURe facilitator test");
console.log(`   Buyer:  ${buyerAccount.address}  (from ${BUYER_KEY_VAR})`);
console.log(`   Seller: ${SELLER_ADDRESS}`);
if (DRY_RUN) console.log("   🧪 DRY_RUN: no on-chain writes");

🚀 x402 exact (Permit2) — EURe facilitator test
   Buyer:  0x553179556FC2A39e535D65b921e01fA995E79101  (from TEST_WALLET_PRIVATE_KEY)
   Seller: 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C


## Network & payment configuration

EURe exists on Base and Base Sepolia only, so there is no Optimism option. Mainnet moves real
money and needs a second, deliberate opt-in for the cells that write on-chain.

In [12]:
const USE_MAINNET = false;           // ⚠️ true = REAL MONEY
const CONFIRM_MAINNET_WRITES = false; // second opt-in for approve + settle on mainnet

const NETWORKS = {
    testnet: { chain: baseSepolia, caip2: "eip155:84532", eure: "0x29F37F6adCa168B79B8d9567eab9BE3fBF21db85", explorer: "https://sepolia.basescan.org/tx/" },
    mainnet: { chain: base,        caip2: "eip155:8453",  eure: "0xbf6e2966A9C3D99C9E4D069E04f7Bdb9C8aa762C", explorer: "https://basescan.org/tx/" },
} as const;
const net = NETWORKS[USE_MAINNET ? "mainnet" : "testnet"];
const EURE = net.eure as `0x${string}`;

// EURe's EIP-712 domain and decimals, read on-chain on both networks (2026-10-04).
const EURE_DOMAIN = { name: "Monerium EURe", version: "1" };
const DECIMALS = 18;
const fmt = (x: bigint) => `${formatUnits(x, DECIMALS)} EURe`;

const PAYMENT_AMOUNT = parseUnits("0.02", DECIMALS);
const mayWrite = !DRY_RUN && (!USE_MAINNET || CONFIRM_MAINNET_WRITES);

const rpcUrl = env[`RPC_URL_EIP155_${net.caip2.split(":")[1]}`] ?? net.chain.rpcUrls.default.http[0];
const publicClient = createPublicClient({ chain: net.chain, transport: http(rpcUrl) });

console.log(`${USE_MAINNET ? "🚨 REAL MONEY on" : "🧪 Testnet:"} ${net.chain.name} (${net.caip2})`);
console.log(`   EURe:    ${EURE}`);
console.log(`   Payment: ${fmt(PAYMENT_AMOUNT)}`);

🚨 REAL MONEY on Base (eip155:8453)
   EURe:    0xbf6e2966A9C3D99C9E4D069E04f7Bdb9C8aa762C
   Payment: 0.02 EURe


## 1. `/supported` — EURe fee and gas sponsoring

The facilitator must advertise `eip2612GasSponsoring`, and list EURe in `facilitatorFees.assets`
with its fee in 18 decimals. The top-level `flatFee` is the nominal fee in 6-decimal units and is
wrong for EURe, so this notebook never uses it.

In [13]:
const FACILITATOR_URL = "http://localhost:8080";
const supported = await (await fetch(`${FACILITATOR_URL}/supported`)).json();

const fees = supported.facilitatorFees;
const eureFee = fees?.assets?.find((a: any) => a.network === net.caip2 && a.symbol === "EURe");
if (!eureFee) {
    throw new Error(
        "The facilitator lists no EURe fee asset — it is an older build. " +
        "Run `npm run build && npm run dev` in x402_facilitator/ on the Permit2 branch.",
    );
}
const facilitatorAddress = fees.recipient as `0x${string}`;
const FEE = BigInt(eureFee.flatFee);
const RECOMMENDED_APPROVAL = BigInt(eureFee.recommended_amount);

console.log(`📡 extensions: ${supported.extensions.join(", ")}`);
console.log(supported.extensions.includes("eip2612GasSponsoring")
    ? "✅ eip2612GasSponsoring advertised"
    : "⚠️ eip2612GasSponsoring NOT advertised");
console.log(`💸 Fee: ${fmt(FEE)} (${FEE} atomic)`);
console.log(`   Recommended approval: ${fmt(RECOMMENDED_APPROVAL)} = ${RECOMMENDED_APPROVAL / FEE} settlements`);
console.log(`   Facilitator: ${facilitatorAddress}`);

📡 extensions: eip2612GasSponsoring, facilitator_fee, facilitatorFees
✅ eip2612GasSponsoring advertised
💸 Fee: 0.01 EURe (10000000000000000 atomic)
   Recommended approval: 1 EURe = 100 settlements
   Facilitator: 0x3F8d2Fb6fEA24E70155bC61471936F3c9C30c206


## 2. Balances before

Besides the three EURe balances, this reads the buyer's EURe **allowance to Permit2**. For a
fresh wallet it is 0, which is why the buyer signs an EIP-2612 permit below.

In [14]:
import { PERMIT2_ADDRESS } from "npm:@x402/evm@^2.26.0";

const ERC20_ABI = [
    { name: "balanceOf", type: "function", stateMutability: "view", inputs: [{ name: "a", type: "address" }], outputs: [{ name: "", type: "uint256" }] },
    { name: "allowance", type: "function", stateMutability: "view", inputs: [{ name: "o", type: "address" }, { name: "s", type: "address" }], outputs: [{ name: "", type: "uint256" }] },
    { name: "approve", type: "function", stateMutability: "nonpayable", inputs: [{ name: "s", type: "address" }, { name: "v", type: "uint256" }], outputs: [{ name: "", type: "bool" }] },
] as const;

const eureBalance = (a: `0x${string}`) =>
    publicClient.readContract({ address: EURE, abi: ERC20_ABI, functionName: "balanceOf", args: [a] });
const eureAllowance = (owner: `0x${string}`, spender: `0x${string}`) =>
    publicClient.readContract({ address: EURE, abi: ERC20_ABI, functionName: "allowance", args: [owner, spender] });

async function snapshot() {
    return {
        buyer: await eureBalance(buyerAccount.address),
        seller: await eureBalance(SELLER_ADDRESS),
        facilitator: await eureBalance(facilitatorAddress),
        buyerPermit2Allowance: await eureAllowance(buyerAccount.address, PERMIT2_ADDRESS as `0x${string}`),
    };
}

const before = await snapshot();
console.log("💰 EURe before:");
console.log(`   Buyer:       ${fmt(before.buyer)}`);
console.log(`   Seller:      ${fmt(before.seller)}`);
console.log(`   Facilitator: ${fmt(before.facilitator)}`);
console.log(`   Buyer → Permit2 allowance: ${fmt(before.buyerPermit2Allowance)}`);
console.log(`⛽ ETH: seller ${formatEther(await publicClient.getBalance({ address: SELLER_ADDRESS }))}, ` +
    `facilitator ${formatEther(await publicClient.getBalance({ address: facilitatorAddress }))}`);

if (before.buyer < PAYMENT_AMOUNT) {
    console.log(`\n⚠️ Buyer has less than ${fmt(PAYMENT_AMOUNT)}. Mint sandbox EURe (see the intro); ` +
        "until then /verify will report permit2_insufficient_balance.");
}

💰 EURe before:
   Buyer:       0.5 EURe
   Seller:      0 EURe
   Facilitator: 0 EURe
   Buyer → Permit2 allowance: 0 EURe
⛽ ETH: seller 0.002698739556519162, facilitator 0.005936660941626891


## 3. Seller approves EURe for the fee

The facilitator pulls its fee from the seller after each settlement, in the token that was paid.
A seller paid in EURe therefore approves **EURe**, and in 18-decimal units: an approval of
`1000000` (1 USDC) covers no EURe fee at all. This cell approves only if the allowance is short.

In [15]:
const currentAllowance = await eureAllowance(SELLER_ADDRESS, facilitatorAddress);
console.log(`🔍 Seller → facilitator allowance: ${fmt(currentAllowance)} = ${currentAllowance / FEE} settlements`);

if (currentAllowance >= FEE) {
    console.log("✅ Enough for at least one settlement — no approval needed");
} else if (!mayWrite) {
    console.log("⏭️  Allowance short, but approve skipped (DRY_RUN or unconfirmed mainnet)");
} else {
    const sellerWallet = createWalletClient({ account: sellerAccount, chain: net.chain, transport: http(rpcUrl) });
    console.log(`📝 approve(${facilitatorAddress}, ${fmt(RECOMMENDED_APPROVAL)})…`);
    const hash = await sellerWallet.writeContract({
        address: EURE, abi: ERC20_ABI, functionName: "approve", args: [facilitatorAddress, RECOMMENDED_APPROVAL],
    });
    const receipt = await publicClient.waitForTransactionReceipt({ hash });
    console.log(`   ✅ ${receipt.status} in block ${receipt.blockNumber}: ${net.explorer}${hash}`);
    const updated = await eureAllowance(SELLER_ADDRESS, facilitatorAddress);
    console.log(`   Allowance now ${fmt(updated)} = ${updated / FEE} settlements`);
}

🔍 Seller → facilitator allowance: 0 EURe = 0 settlements
📝 approve(0x3F8d2Fb6fEA24E70155bC61471936F3c9C30c206, 1 EURe)…
   ✅ success in block 52436470: https://basescan.org/tx/0xb5bc51ee508b4e1ceb8fcea1e606c8ba9838cbf69c64b5edadc660501a38bf55
   Allowance now 0 EURe = 0 settlements


## 4. Buyer signs the Permit2 payment

`ExactEvmScheme` (the SDK's buyer-side client) picks Permit2 because the requirements say
`assetTransferMethod: "permit2"`. The mock 402 declares `eip2612GasSponsoring`, and `x402Client`
hands that to the scheme. The scheme then reads the buyer's Permit2 allowance and EURe `nonces`
on-chain, and signs an EIP-2612 permit if the allowance is short.

In [16]:
import { x402Client } from "npm:@x402/fetch@^2.26.0";
import type { Network } from "npm:@x402/core@^2.26.0";
import { ExactEvmScheme } from "npm:@x402/evm@^2.26.0/exact/client";

const buyerSigner = {
    address: buyerAccount.address,
    signTypedData: (msg: any) => buyerAccount.signTypedData(msg),
    readContract: (args: any) => publicClient.readContract(args),
};

function buyerClient() {
    const client = new x402Client();
    client.register("eip155:*" as Network, new ExactEvmScheme(buyerSigner as any));
    // The SDK's default spend controls only know its own registry's tokens.
    client.setSpendControls({ allowedAssets: [{ network: net.caip2 as `${string}:${string}`, asset: EURE }] });
    return client;
}

function requirementsFor(payTo: `0x${string}`) {
    return {
        scheme: "exact",
        network: net.caip2 as Network,
        amount: PAYMENT_AMOUNT.toString(),
        asset: EURE,
        payTo,
        maxTimeoutSeconds: 300,
        extra: { ...EURE_DOMAIN, assetTransferMethod: "permit2" },
    };
}

function paymentRequiredFor(payTo: `0x${string}`) {
    return {
        x402Version: 2,
        resource: { url: "https://example.com/eure-test", description: "EURe Permit2 test", mimeType: "text/plain" },
        accepts: [requirementsFor(payTo)],
        extensions: { eip2612GasSponsoring: {} },
    };
}

const paymentRequirements = requirementsFor(SELLER_ADDRESS);
const paymentPayload: any = await buyerClient().createPaymentPayload(paymentRequiredFor(SELLER_ADDRESS) as any);

const auth = paymentPayload.payload.permit2Authorization;
console.log("✍️  permit2Authorization:");
console.log(`   from:       ${auth.from}`);
console.log(`   spender:    ${auth.spender}  (x402 exact Permit2 proxy)`);
console.log(`   witness.to: ${auth.witness.to}  (= seller)`);
console.log(`   amount:     ${fmt(BigInt(auth.permitted.amount))}`);
const permit = paymentPayload.extensions?.eip2612GasSponsoring?.info;
console.log(permit
    ? `🔏 EIP-2612 permit for Permit2: ${fmt(BigInt(permit.amount))}, nonce ${permit.nonce}`
    : "ℹ️  No EIP-2612 permit — the buyer already approved Permit2 enough");

✍️  permit2Authorization:
   from:       0x553179556FC2A39e535D65b921e01fA995E79101
   spender:    0x402085c248EeA27D92E8b30b2C58ed07f9E20001  (x402 exact Permit2 proxy)
   witness.to: 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C  (= seller)
   amount:     0.02 EURe
🔏 EIP-2612 permit for Permit2: 0.02 EURe, nonce 0


## 5. Safety check: a payment to someone else is refused

The buyer signs the same payment but to a different address. `/verify` checks it against the
**real** requirements (`payTo` = the seller) and must refuse it because the Permit2 witness pins the
recipient. Nothing goes on-chain.

In [17]:
const OTHER = "0x000000000000000000000000000000000000dEaD" as `0x${string}`;
const misdirected: any = await buyerClient().createPaymentPayload(paymentRequiredFor(OTHER) as any);
misdirected.accepted = paymentRequirements; // claim the seller's terms, keep the signature to OTHER

const mismatch = await (await fetch(`${FACILITATOR_URL}/verify`, {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ paymentPayload: misdirected, paymentRequirements }),
})).json();
console.log(mismatch.invalidReason === "invalid_permit2_recipient_mismatch"
    ? "✅ refused: invalid_permit2_recipient_mismatch"
    : `❌ unexpected: ${JSON.stringify(mismatch)}`);

✅ refused: invalid_permit2_recipient_mismatch


## 6. `/verify`

Expected: `isValid: true` and `remainingSettlements` counted in EURe (100 for a 1-EURe approval).
With an unfunded buyer, expect `permit2_insufficient_balance`: the facilitator simulated
`settleWithPermit` against the live proxy and only the balance failed.

In [18]:
const verifyResult = await (await fetch(`${FACILITATOR_URL}/verify`, {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ paymentPayload, paymentRequirements }),
})).json();
console.log(JSON.stringify(verifyResult, null, 2));
if (verifyResult.isValid) {
    console.log(`✅ valid — ${verifyResult.remainingSettlements} settlements left on the seller's EURe allowance`);
} else if (verifyResult.invalidReason === "permit2_insufficient_balance") {
    console.log("ℹ️  Wiring accepted; the buyer just has no EURe yet (mint it in the Monerium sandbox).");
} else {
    console.log(`❌ ${verifyResult.invalidReason}`);
}

{
  "isValid": true,
  "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
  "remainingSettlements": 100
}
✅ valid — 100 settlements left on the seller's EURe allowance


## 7. `/settle`

> ⚠️ Real on-chain transaction. The facilitator calls `settleWithPermit` on the proxy (permit and
> transfer in one transaction), then pulls the 0.01 EURe fee from the seller.

In [19]:
let settleResult: any = null;
if (!verifyResult.isValid) {
    console.log("⏭️  Skipped — /verify did not pass");
} else if (!mayWrite) {
    console.log("⏭️  Skipped (DRY_RUN or unconfirmed mainnet)");
} else {
    settleResult = await (await fetch(`${FACILITATOR_URL}/settle`, {
        method: "POST", headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ paymentPayload, paymentRequirements }),
    })).json();
    console.log(JSON.stringify(settleResult, null, 2));
    const info = settleResult.extensions?.facilitatorFees?.info;
    if (settleResult.success) {
        console.log(`✅ settled: ${net.explorer}${settleResult.transaction}`);
        console.log(`💸 fee assessed: ${fmt(BigInt(info?.facilitatorFeePaid ?? "0"))}, collection ${info?.collection?.status}`);
        if (settleResult.fee?.txHash) console.log(`   fee tx: ${net.explorer}${settleResult.fee.txHash}`);
    }
}

{
  "success": true,
  "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
  "transaction": "0x325ee28be01911b4e89eab640e941714917cae1cc29b30bc00d1a99e22d5def6",
  "network": "eip155:8453",
  "fee": {
    "collected": true,
    "status": "collected",
    "txHash": "0xe5b3b873b0d3ddde5ad54a47ad0f0ee5ce897cf5ee3dd8d73aa68615da9c433a"
  },
  "extensions": {
    "facilitatorFees": {
      "info": {
        "version": "1",
        "facilitatorFeePaid": "10000000000000000",
        "asset": "eip155:8453/erc20:0xbf6e2966A9C3D99C9E4D069E04f7Bdb9C8aa762C",
        "model": "flat",
        "collection": {
          "status": "collected",
          "txHash": "0xe5b3b873b0d3ddde5ad54a47ad0f0ee5ce897cf5ee3dd8d73aa68615da9c433a"
        }
      }
    }
  }
}
✅ settled: https://basescan.org/tx/0x325ee28be01911b4e89eab640e941714917cae1cc29b30bc00d1a99e22d5def6
💸 fee assessed: 0.01 EURe, collection collected
   fee tx: https://basescan.org/tx/0xe5b3b873b0d3ddde5ad54a47ad0f0ee5ce897cf5ee3dd8d73aa6861

## 8. Balances after

Expected deltas: buyer −0.02, seller +0.02 − 0.01 fee, facilitator +0.01. The buyer's Permit2
allowance is spent by the transfer, so it returns to where it started.

In [20]:
if (settleResult?.success) {
    const after = await snapshot();
    const row = (label: string, delta: bigint, expected: bigint) =>
        console.log(`   ${label.padEnd(12)} ${fmt(delta).padStart(16)}  expected ${fmt(expected).padStart(16)}  ${delta === expected ? "✅" : "❌"}`);
    console.log("📊 EURe deltas:");
    row("Buyer", after.buyer - before.buyer, -PAYMENT_AMOUNT);
    row("Seller", after.seller - before.seller, PAYMENT_AMOUNT - FEE);
    row("Facilitator", after.facilitator - before.facilitator, FEE);
    console.log(`   Buyer → Permit2 allowance: ${fmt(before.buyerPermit2Allowance)} → ${fmt(after.buyerPermit2Allowance)}`);
} else {
    console.log("⏭️  No settlement in this run");
}

📊 EURe deltas:
   Buyer              -0.02 EURe  expected       -0.02 EURe  ✅
   Seller              0.01 EURe  expected        0.01 EURe  ✅
   Facilitator         0.01 EURe  expected        0.01 EURe  ✅
   Buyer → Permit2 allowance: 0 EURe → 0 EURe


## Summary

| Section | What it proves |
|---|---|
| 1 | The facilitator advertises `eip2612GasSponsoring` and an 18-decimal EURe fee |
| 3 | The seller's EURe fee allowance, in EURe units |
| 4 | The SDK client builds a Permit2 payload with an EIP-2612 permit for a fresh buyer |
| 5 | A payment signed to another recipient is refused (`invalid_permit2_recipient_mismatch`) |
| 6 | The live proxy and token accept the payment (or only the balance is missing) |
| 7–8 | One `settleWithPermit` moves the EURe, and the fee pull takes 0.01 EURe from the seller |